# Used to look what is in NVME reads

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns
import gc


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
# app_name = "montage-pegasus-2mass-2deg-4node" #cosmoflow cm1
# app_name = "1000_genome_pegasus_node_16"
# app_name = "cm1"
# app_name = "deepspeed-dlio-step100"
app_name = "deepspeed-dlio-scr-step100"
# app_name = "bert"
# app_name = "unet3d"
# app_name = "resnet50"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint"

condition_fn = None #

if app_name == "cm1":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/cm1/APP/node-32/v1/COMPACT/*.pfw.gz"
    # cp_dir = "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "deepspeed-dlio-step100":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-scr-step100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "deepspeed-dlio-scr-step100":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-scr-step100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "montage-mpi-2mass-7deg":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/mpi-2mass-7deg/node-16/v1/RAW/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name =="montage-pegasus-2mass-2deg-4node":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-4/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-4/v1/RAW/*.pfw.gz"
    filename = "/p/lustre3/pandey2/logs/RAW_copy/montage-pegasus-2mass-2deg-4node/RAW/*.pfw.gz" # Raw Copied
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "resnet50":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/resnet50/dlio-v100/node-4/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "unet3d":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/unet3d/dlio-v100/node-16/v2/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "bert":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/bert/v100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "1000_genome_pegasus_node_16":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/1000-genome/pegasus/node-16/v3/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"
    
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [16:14:41] Initialized Client with 288 workers and link http://134.9.71.20:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]
[INFO] [16:14:59] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]


In [3]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [4]:
def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 
    return d

load_cols = {'size': "int64[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}

In [5]:
analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)

[INFO] [16:15:45] Created index for 16 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [16:15:45] Total size of all files are <dask.bag.core.Item object at 0x155541740610> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [16:15:45] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [16:15:48] Loading 8410 batches out of 16 files and has 137639290 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [16:21:33] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [16:21:33] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


In [6]:
analyzer.summary()

[INFO] [19:04:17] Total number of events in the workload are 409777294 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:623]
[WARNING] [19:04:22] The max io_time 24179875 exceeds the time_granularity 1000000.0. Please adjust the time_granularity to 48e6 and rerun the analyzer. [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:531]


╭──────────────────────────────────────────────────── Summary ────────────────────────────────────────────────────╮
│  Allocation    Scheduler Allocation Details                                                                     │
│                ├── Nodes: 16                                                                                    │
│                ├── Processes: 6098                                                                              │
│                ├── Thread allocations across nodes (includes dynamically created threads)                       │
│                │   ├── Compute: 0                                                                               │
│                │   └── I/O: 6098                                                                                │
│                └── Events Recorded: 410M                                                                        │
│  Dataset       Description of Dataset Used                                                                      │
│                └── Files: 25426951                                                                              │
│  I/O Behavior  Behavior of Application                                                                          │
│                ├── Split of Time in application                                                                 │
│                │   ├── Total Time: 5771.193 sec                                                                 │
│                │   └── Overall I/O: 2576.957 sec                                                                │
│                └── Metrics by function                                                                          │
│                    ├── Function       |count |                  size                   |                        │
│                    ├──                |      |min   |25    |mean  |median|75    |max   |                        │
│                    ├── __lxstat64     |221M  |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── lseek64        |109M  |NA    |15KB  |110KB |38KB  |293KB |369KB |                        │
│                    ├── read           |43M   |NA    |6KB   |52KB  |64KB  |64KB  |4MB   |                        │
│                    ├── opendir        |174K  |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── open64         |7M    |7     |8     |8     |8     |9     |9     |                        │
│                    ├── __fxstat64     |7M    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── close          |7M    |NA    |NA    |NA    |NA    |NA    |NA    |                        │
│                    ├── readlink       |3K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── __xstat64      |8M    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── rmdir          |7K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── access         |10K   |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── mkdir          |2K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── chmod          |7M    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── open           |216K  |-1    |8     |32    |12    |31    |55    |                        │
│                    ├── fork           |11K   |NA    |9KB   |841KB |934KB |2MB   |4MB   |                        │
│                    ├── __xstat        |412K  |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── unlink         |7K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── __fxstat       |14K   |NA    |n

In [52]:
dir_w = "/p/lustre3/pandey2/logs/Results_Checkpoint/fhash/"+app_name+"/"
os.makedirs(dir_w, exist_ok=True)
analyzer.file_hash.reset_index().to_parquet(f'{dir_w}',engine='pyarrow',write_index=False)

In [ ]:
analyzer.file_hash.reset_index().[[""]]("mount_point").count().compute()


,hash,name,pid,tid,hhash
mount_point,,,,,
./,1,1,1,1,1
/collab,1,1,1,1,1
/collab/usr,1,1,1,1,1
/collab/usr/gapps,912,912,912,912,912
/dev,23,23,23,23,23
/dev/shm,391824,391824,391824,391824,391824
/etc/crypto-policies,2,2,2,2,2
/etc/libfabric.conf,1,1,1,1,1
/etc/libibverbs.d,16,16,16,16,16


In [6]:
if app_name in ["deepspeed-dlio-step100","deepspeed-dlio-scr-step100","resnet50","unet3d"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']]
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name == "montage-pegasus-2mass-2deg-4node":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name in ["1000_genome_pegasus_node_16"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]


elif app_name == "bert":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name in ["cm1"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]




In [7]:
data_df.head()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,id
1,read,POSIX,1048576,1540455127,1540456291,1164,1540,/l/ssd,corona200,1
4,read,POSIX,1048576,1540478490,1540480316,1826,1540,/l/ssd,corona200,4
7,read,POSIX,1048576,1540500324,1540501915,1591,1540,/l/ssd,corona200,7
10,read,POSIX,1048576,1540520922,1540521582,660,1540,/l/ssd,corona200,10
13,read,POSIX,1048576,1540541819,1540543769,1950,1540,/l/ssd,corona200,13


In [7]:
# Intermediate Steps Required 
# Step 1: Filter correct rows and categorize based on the size

bins = [2**i for i in range(0, 21)] + [float('inf')] 
bin_labels = [
    f"2^{i} - 2^{i+1} B" if i < 10 else
    f"2^{i-10} KiB - 2^{i-9} KiB" if i < 20 else
    "1 MiB+"
    for i in range(0, 21)
]

def categorize_sizes(df):
    df['size_category'] = pd.cut(df['size'], bins=bins, labels=bin_labels)
    return df

def categorize_sizes_metadata(df):
    df['size_category'] = "size"
    return df

data_df = data_df[data_df["size"] > 1]
data_df = data_df.map_partitions(categorize_sizes)
data_df['size_category'] = data_df['size_category'].astype('string[pyarrow]')

metadata_df = metadata_df.query("dur > 0")
metadata_df["size"] = 0 
metadata_df = metadata_df.map_partitions(categorize_sizes_metadata)
metadata_df['size_category'] = metadata_df['size_category'].astype('string[pyarrow]')


# # step 2: Filter only interesting mount points
# # if computationally expensive and only few 
# if app_name == "resnet500":
#     mount_point_list = ["/p/lustre3","/dev","/proc","/sys","/dev","/dev/shm","/usr/workspace"]
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# # if app_name == "deepspeed-dlio-scr-step100":
# #     mount_point_list = ['/l/ssd', '/p/lustre3','/usr/workspace', '/usr/WS2', '/sys', '/collab/usr/gapps', '/dev/shm','/dev','/proc', '/usr/tce','/usr/share','/var/tmp', '/g/g92']
# #     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
# #     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# if app_name == 'montage-pegasus-2mass-2deg-4node':
#     mount_point_list = ['/p/lustre3', '/proc', '/usr/WS2', '/dev/shm', '/dev']
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# if app_name == 'cm1':
#     mount_point_list =  ['/dev','/dev/shm', '/etc/libibverbs.d', '/p/lustre3','/sys','/tmp', '/usr/tce','/var/tmp', '/proc', '/usr/lib64', '/etc/psm3.conf', '/var']  
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# if app_name == '1000_genome_pegasus_node_16':
#     mount_point_list =  ['/p/lustre3', '/usr/WS2', '/usr/workspace', '/dev', '/proc', '/sys']
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]
    


 

In [8]:
data_df.head()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,id,size_category
33285,read,POSIX,3130,21087678,21087816,138,0,/p/lustre3,corona197,33285,2^1 KiB - 2^2 KiB
33304,read,POSIX,1272,21114029,21114407,378,0,/p/lustre3,corona197,33304,2^0 KiB - 2^1 KiB
33341,read,POSIX,3422,21163963,21164069,106,0,/p/lustre3,corona197,33341,2^1 KiB - 2^2 KiB
33370,read,POSIX,2289,21167470,21167744,274,0,/p/lustre3,corona197,33370,2^1 KiB - 2^2 KiB
33451,read,POSIX,206,21400421,21400920,499,0,/p/lustre3,corona197,33451,2^7 - 2^8 B


## Data Interference calculation

In [8]:
#DATA Interference Computation
IFCalculator = DFGrepInterferencePartitionBased(data_df, app_name=app_name, operation="data", cp_dir=cp_dir, existing=False)


In [9]:
IFCalculator.computeDegree()


In [10]:
IFCalculator.computeInterferenceData() 

[INFO] [18:26:32] Computing duration of minimum degree event for all degrees. [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph1.py:775]
[INFO] [18:43:21] Computing IF for all  [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph1.py:788]


In [ ]:
IFCalculator.inter.query()

In [12]:
IFCalculator.inter.query('interference > 0 and deg_caller > 1').groupby('mount_point').count().compute()

,name,cat,size,size_category,ts,te,dur,trange,hostname,deg_caller,deg_other,min_dur,interference
mount_point,,,,,,,,,,,,,
/dev,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506
/dev/shm,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868
/p/lustre3,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061
/proc,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766
/sys,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830
/tmp,49,49,49,49,49,49,49,49,49,49,49,49,49
/usr/tce,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725
/var,24,24,24,24,24,24,24,24,24,24,24,24,24
/var/tmp,310,310,310,310,310,310,310,310,310,310,310,310,310


In [12]:
IFCalculator.inter.query('interference > 0 and deg_caller > 1').groupby('mount_point').count().compute()

KeyboardInterrupt: 

In [11]:
IFCalculator.write_checkpoint(id='inter', cp_dir=cp_dir)

In [15]:
IFCalculator.inter.query('interference > 0 and deg_caller > 1').compute()

,name,cat,size,size_category,ts,te,dur,trange,mount_point,hostname,deg_caller,deg_other,min_dur,interference
3539,read,POSIX,5412,2^2 KiB - 2^3 KiB,32316195,32316948,753,1,/collab/usr/gapps,corona202,2,1.0,4.0,0.005312
3868,read,POSIX,289,2^8 - 2^9 B,31650827,31651360,533,1,/usr/WS2,corona202,2,1.0,2.0,0.003752
3896,read,POSIX,22552,2^4 KiB - 2^5 KiB,31730406,31732770,2364,1,/usr/WS2,corona202,2,1.0,5.0,0.002115
3907,read,POSIX,8885,2^3 KiB - 2^4 KiB,31751905,31752294,389,1,/usr/WS2,corona202,2,1.0,5.0,0.012853
3918,read,POSIX,8192,2^2 KiB - 2^3 KiB,31757693,31757697,4,1,/usr/WS2,corona202,2,1.0,5.0,1.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3677,read,POSIX,730,2^9 - 2^10 B,24208616,24209336,720,0,/usr/WS2,corona193,2,1.0,3.0,0.004167
3691,read,POSIX,420,2^8 - 2^9 B,24259458,24259579,121,0,/usr/WS2,corona193,2,1.0,3.0,0.024793
3705,read,POSIX,27332,2^4 KiB - 2^5 KiB,24302613,24303478,865,0,/usr/WS2,corona193,2,1.0,3.0,0.003468
3730,read,POSIX,7845,2^2 KiB - 2^3 KiB,24373964,24374258,294,0,/usr/WS2,corona193,2,1.0,3.0,0.010204


# Metadata Interference Computation

In [16]:
IFCalculator_Metadata = DFGrepInterference(metadata_df, app_name=app_name, operation="metadata", cp_dir=cp_dir, existing=False)
# IFCalculator_Metadata.get_degree()
# IFCalculator_Metadata.compute_interference()
# IFCalculator_Metadata.write_checkpoint(id="inter_metadata", cp_dir = cp_dir)

In [17]:
IFCalculator_Metadata.get_degree()

In [ ]:
IFCalculator_Metadata.compute_interference()

In [ ]:
IFCalculator_Metadata.write_checkpoint(id="inter_metadata", cp_dir = cp_dir)

,name,pid,tid,hhash
hash,,,,
1.181405777718347e+19,corona188,1027673,1027673,11814057777183469021
6.968018510730724e+18,corona239,2055310,2055310,6968018510730723892
6.598453666498801e+18,corona234,2382606,2382606,6598453666498800848
1.0812942414789165e+19,corona236,269213,269213,10812942414789164657
1.1063386352187771e+18,corona238,2440296,2440296,1106338635218777131


# Burstiness Computation

In [8]:
delta = data_df.dur.max().compute()
BurstCalculator = DFGrepBurstiness(data_df, app_name=app_name, operation="data", delta = delta, cp_dir=cp_dir, existing=False)
BurstCalculator.compute_burstiness()
# BurstCalculator.write_checkpoint(id="ddf_bur",cp_dir=cp_dir)

In [9]:
BurstCalculator.write_checkpoint(id="ddf_bur",cp_dir=cp_dir)

In [12]:
BurstCalculator.ddf_bur.head()

,id,name,cat,size,ts,te,dur,trange,mount_point,hostname,group_key,b_id
0,443935,read,POSIX,65536,3302000003,3302000023,20,3302,/usr/WS2,corona243,"(3302, '/usr/WS2')",443935
1,72186,read,POSIX,65536,3302000073,3302000096,23,3302,/usr/WS2,corona188,"(3302, '/usr/WS2')",443935
2,443940,read,POSIX,65536,3302000080,3302000098,18,3302,/usr/WS2,corona243,"(3302, '/usr/WS2')",443935
3,72189,read,POSIX,65536,3302000121,3302000144,23,3302,/usr/WS2,corona188,"(3302, '/usr/WS2')",443935
4,443943,read,POSIX,65536,3302000123,3302000138,15,3302,/usr/WS2,corona243,"(3302, '/usr/WS2')",443935


<!-- delta = data_df.dur.max().compute() -->
